In [2]:
import pandas as pd
import numpy as np
import scipy.sparse as sparse
import joblib
import implicit
import os

from sklearn.preprocessing import LabelEncoder

os.makedirs('../models', exist_ok=True)

In [3]:
ratings_filtered = pd.read_parquet('../data/ratings_filtered.parquet')
movies_df        = pd.read_csv('../data/movies.csv')

print(f'Ratings: {len(ratings_filtered):,}')

Ratings: 24,644,928


In [4]:
user_enc  = LabelEncoder()
movie_enc = LabelEncoder()

ratings_filtered = ratings_filtered.copy()

ratings_filtered['user_idx']  = user_enc.fit_transform(ratings_filtered['userId'])
ratings_filtered['movie_idx'] = movie_enc.fit_transform(ratings_filtered['movieId'])

n_users  = ratings_filtered['user_idx'].max() + 1
n_movies = ratings_filtered['movie_idx'].max() + 1

print(f'Matrix size: {n_users:,} users x {n_movies:,} movies')

Matrix size: 162,540 users x 13,176 movies


In [5]:
import scipy.sparse as sparse
user_item = sparse.csr_matrix((
    ratings_filtered['rating'],
    (
        ratings_filtered['user_idx'].values,
        ratings_filtered['movie_idx'].values
    )
    ),shape = (n_users , n_movies)
    ) 

In [6]:
model = implicit.als.AlternatingLeastSquares(
    factors=64,
    regularization=0.1,
    iterations=20,
    calculate_training_loss=True,
    random_state=42
)

model.fit(user_item)

100%|██████████| 20/20 [01:57<00:00,  5.87s/it, loss=0.0148]


In [7]:
def ALS_recommend(real_id):
    user_idx = user_enc.transform([real_id])[0]

    movie_indices , scores = model.recommend(
        userid = user_idx,
        user_items = user_item[user_idx],
        N = 10
    )
    real_movie_ids = movie_enc.inverse_transform(movie_indices)

    recommended_df = pd.DataFrame({
         'movieId': real_movie_ids,
         'score': scores
})

    final_recommendations = pd.merge(recommended_df, movies_df, on='movieId')
    return final_recommendations

ALS_recommend(3)

,movieId,score,title,genres
0,4370,1.177602,A.I. Artificial Intelligence (2001),Adventure|Drama|Sci-Fi
1,8972,1.152896,National Treasure (2004),Action|Adventure|Drama|Mystery|Thriller
2,608,1.090408,Fargo (1996),Comedy|Crime|Drama|Thriller
3,115149,1.075915,John Wick (2014),Action|Thriller
4,53322,1.068200,Ocean's Thirteen (2007),Crime|Thriller
5,30812,1.066138,"Aviator, The (2004)",Drama
6,5010,1.021524,Black Hawk Down (2001),Action|Drama|War
7,4270,1.010739,"Mummy Returns, The (2001)",Action|Adventure|Comedy|Thriller
8,119145,1.007584,Kingsman: The Secret Service (2015),Action|Adventure|Comedy|Crime
9,122904,0.995282,Deadpool (2016),Action|Adventure|Comedy|Sci-Fi


In [8]:
joblib.dump(model,     '../models/als_model.pkl')
joblib.dump(user_enc,  '../models/user_encoder.pkl')
joblib.dump(movie_enc, '../models/movie_encoder.pkl')
sparse.save_npz('../models/user_item_matrix.npz', user_item)

ratings_filtered.to_parquet('../data/ratings_encoded.parquet', index=False)

print('Saved:')
print('  ../models/als_model.pkl')
print('  ../models/user_encoder.pkl')
print('  ../models/movie_encoder.pkl')
print('  ../models/user_item_matrix.npz')
print('  ../data/ratings_encoded.parquet')

Saved:
  ../models/als_model.pkl
  ../models/user_encoder.pkl
  ../models/movie_encoder.pkl
  ../models/user_item_matrix.npz
  ../data/ratings_encoded.parquet
